# 第 01 章 数据导入与对象检查

## 学习目标

读取两个样本，保留样本来源，理解细胞、基因和表达矩阵之间的对应关系。

## 为什么做与怎样做

使用 read_10x_h5 读取 Gene Expression 特征，对基因名去重后按共有基因合并，用 samples 列保留来源。

前置章节：00。运行前请完成项目环境准备。


In [1]:
from __future__ import annotations
from pathlib import Path
import os
import sys
ROOT = Path(os.environ.get("SC_COURSE_ROOT", Path.cwd())).resolve()
while not (ROOT / "config/course.json").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config/course.json").is_file():
    raise RuntimeError("请从课程目录或其 notebooks/exercises 目录运行。")
os.environ["CELLTYPIST_FOLDER"] = str(ROOT / ".runtime/celltypist")
os.environ["MPLCONFIGDIR"] = str(ROOT / ".runtime/matplotlib")
sys.path.insert(0, str(ROOT / "tools"))
from course_runtime import start_chapter, marker_sets, scaled_view
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
from scipy.sparse import csr_matrix
from scipy.stats import median_abs_deviation
ctx = start_chapter("01")


第 01 章：数据导入与对象检查
结果目录：results/01_data_import/20260915T072732_9b3fa3


## 01.1 数据导入

本课程使用 NeurIPS 2021 单细胞多组学基准数据集中的两个 10X Gene Expression 文件：s1d1 与 s1d3。这里分析 RNA 计数，不分析 ATAC。样本标识保留为 samples，不把样本差异自动解释为纯技术效应。

来源：[数据集页面](https://figshare.com/articles/dataset/NeurIPS_2021_Benchmark_dataset/22716739)、[Scanpy 教程](https://scanpy.scverse.org/en/stable/tutorials/basics/clustering.html)。

课程包已包含所需 H5 数据。需要核对来源时，可访问上面的数据集页面。

In [2]:
# 功能说明：下载并读取两个样本的 10x HDF5 计数矩阵，合并为单一 AnnData。
# 运行目的：构建包含所有细胞的综合对象并保留样本来源标签，用于后续 QC 与分析。
# 变量/函数/参数解析（逐项）：
# - samples(dict)：样本 ID 到文件名的映射；键为 "s1d1"、"s1d3"，值为各自的 .h5 文件名。
# - adatas(dict)：用于暂存每个样本的 AnnData。
# - for sample_id, filename in samples.items()：遍历样本映射。
#   - EXAMPLE_DATA.fetch(filename)：从缓存/远程下载对应文件并返回本地路径。
#   - sc.read_10x_h5(path)：读取 10x 格式 HDF5 计数矩阵为 AnnData。
#   - sample_adata.var_names_make_unique()：唯一化基因名，避免重复导致冲突。
#   - adatas[sample_id] = sample_adata：以样本 ID 为键保存。
# - ad.concat(adatas, label="samples")：按细胞维度拼接多个 AnnData，并在 `obs['sample']` 写入来源标签。join='inner'  ['inner', 'outer'] (默认值: 'inner')指定拼接时数值的对齐方式。若选择“outer”，则取其他轴的并集；若选择“inner”，则取交集。
# - adata.obs_names_make_unique()：唯一化细胞名，避免重复。
# - print(adata.obs["samples"].value_counts())：打印各样本细胞数量统计。
# 数据流程：
# - 输入：两个 10x .h5 计数文件。
# - 输出：合并后的 `adata`（细胞为两样本并集，基因为并集）。

samples = ctx.config["samples"]
adatas = {}

for sample_id, filename in samples.items():
    #path = EXAMPLE_DATA.fetch(filename)
    path = ROOT / "data" / "h5" / filename
    sample_adata = sc.read_10x_h5(path)
    sample_adata.var_names_make_unique()
    adatas[sample_id] = sample_adata

adata = ad.concat(adatas, label="samples",join='inner')
adata.obs_names_make_unique()
print(adata.obs["samples"].value_counts())


/data/home/heqingchuan/workdir/19_方超老师合作_单细胞_gpt6/sc_RNA/sc_RNA_basic_course/.envs/sc_rna/lib/python3.12/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


/data/home/heqingchuan/workdir/19_方超老师合作_单细胞_gpt6/sc_RNA/sc_RNA_basic_course/.envs/sc_rna/lib/python3.12/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


/data/home/heqingchuan/workdir/19_方超老师合作_单细胞_gpt6/sc_RNA/sc_RNA_basic_course/.envs/sc_rna/lib/python3.12/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


/data/home/heqingchuan/workdir/19_方超老师合作_单细胞_gpt6/sc_RNA/sc_RNA_basic_course/.envs/sc_rna/lib/python3.12/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


samples
s1d1    8785
s1d3    8340
Name: count, dtype: int64


/data/home/heqingchuan/workdir/19_方超老师合作_单细胞_gpt6/sc_RNA/sc_RNA_basic_course/.envs/sc_rna/lib/python3.12/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


读取数据后，scanpy 会显示警告，指出并非所有变量名称都是唯一的。 这表明某些变量（此处为基因）出现不止一次，这可能会导致下游分析任务出现错误或意外行为。 我们执行建议的函数 var_names_make_unique()，它通过向每个重复的索引元素附加一个数字字符串使变量名称唯一：‘1’，‘2’ 等。

In [3]:
# 功能说明：查看 AnnData 对象的摘要信息。
# 运行目的：检查数据加载是否成功，查看细胞数（n_obs）、基因数（n_vars）及已有的注释信息。
# 详细代码解析：
# 1. `adata`
#    - 在 Jupyter Notebook 中直接输入变量名，会调用其 `__repr__` 方法，打印对象的概览。
#    - 输出通常包含：
#      - `n_obs × n_vars`: 细胞数 × 基因数。
#      - `obs`: 细胞的观测注释（如样本来源）。
#      - `var`: 基因的特征注释（如基因名）。
adata

AnnData object with n_obs × n_vars = 17125 × 36601
    obs: 'samples'

In [4]:
print(adata.X)

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 26550469 stored elements and shape (17125, 36601)>
  Coords	Values
  (0, 43)	3.0
  (0, 59)	1.0
  (0, 60)	1.0
  (0, 62)	1.0
  (0, 72)	1.0
  (0, 73)	1.0
  (0, 86)	1.0
  (0, 104)	3.0
  (0, 145)	1.0
  (0, 170)	14.0
  (0, 184)	1.0
  (0, 187)	1.0
  (0, 208)	3.0
  (0, 219)	6.0
  (0, 237)	1.0
  (0, 238)	1.0
  (0, 241)	1.0
  (0, 297)	1.0
  (0, 354)	1.0
  (0, 359)	1.0
  (0, 370)	1.0
  (0, 371)	1.0
  (0, 401)	1.0
  (0, 434)	1.0
  (0, 474)	1.0
  :	:
  (17124, 36383)	1.0
  (17124, 36384)	1.0
  (17124, 36390)	1.0
  (17124, 36396)	1.0
  (17124, 36400)	1.0
  (17124, 36401)	82.0
  (17124, 36404)	1.0
  (17124, 36428)	1.0
  (17124, 36430)	1.0
  (17124, 36432)	1.0
  (17124, 36450)	6.0
  (17124, 36494)	1.0
  (17124, 36559)	34.0
  (17124, 36560)	13.0
  (17124, 36561)	40.0
  (17124, 36562)	62.0
  (17124, 36563)	1.0
  (17124, 36564)	82.0
  (17124, 36565)	62.0
  (17124, 36566)	53.0
  (17124, 36567)	2.0
  (17124, 36568)	24.0
  (17124, 36569)	6.0
  (1

In [5]:
adata.obs

,samples
AAACCCAAGGATGGCT-1,s1d1
AAACCCAAGGCCTAGA-1,s1d1
AAACCCAAGTGAGTGC-1,s1d1
AAACCCACAAGAGGCT-1,s1d1
AAACCCACATCGTGGC-1,s1d1
...,...
TTTGTTGAGAGTCTGG-1,s1d3
TTTGTTGCAGACAATA-1,s1d3
TTTGTTGCATGTTACG-1,s1d3
TTTGTTGGTAGTCACT-1,s1d3


每个样本约有 8,000 个条形码；实际细胞数和基因数以本次读取结果为准。

## 保存本章结果

保存表格、参数摘要和可供后续章节读取的数据。

In [6]:
ctx.table("sample_counts", adata.obs["samples"].value_counts().rename("n_cells"))
ctx.finish(adata, {"samples": adata.obs["samples"].value_counts().to_dict()})

本章计算完成。请阅读本次图表和表格，再更新本章解读。


## 结果阅读与思考

请打开本次结果目录中的 summary.json、tables 和 figures。将目的、方法、结果和解释写入本章报告源稿，再更新 Word。

思考题：样本合并以后，为什么细胞索引仍然需要唯一？

运行与报告操作见课程根目录的 99_运行与AI协作指南.md。